# 10장. 분류 분석으로 주문 취소 여부 예측하기

이 노트북은 완료 주문과 취소 주문만 사용해 `is_cancelled`를 예측하는 이진 분류 실습입니다.

## 학습 목표

- `completed=0`, `cancelled=1`로 타깃 범위를 명확하게 정의합니다.
- `refunded`와 기타 상태를 이진 분류 대상에서 제외합니다.
- 병합의 키 관계, 행 수, 미매칭을 검증합니다.
- train, validation, test를 분리합니다.
- Dummy 기준 모델, Logistic Regression, Random Forest를 비교합니다.
- validation에서 모델과 임계값을 선택하고 test는 최종 평가에 한 번만 사용합니다.
- 예측 결과에서 고객명 등 불필요한 개인정보를 제외합니다.


## 1. 프로젝트 루트와 실행 환경 설정

전처리 파일이 없다면 프로젝트 루트에서 먼저 다음 명령을 실행하세요.

```bash
python scripts/preprocess_data.py
```


In [1]:
from pathlib import Path
import sys

import pandas as pd


def find_project_root(start_path):
    start_path = Path(start_path).resolve()
    for candidate in [start_path, *start_path.parents]:
        if (candidate / 'requirements.txt').exists() and (candidate / 'scripts').exists():
            return candidate
    raise FileNotFoundError('프로젝트 루트 폴더를 찾을 수 없습니다.')


PROJECT_ROOT = find_project_root(Path.cwd())
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
REPORT_DIR = PROJECT_ROOT / 'reports'
REPORT_DIR.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('Python 실행 파일:', sys.executable)
print('프로젝트 루트:', PROJECT_ROOT)
print('전처리 데이터 폴더:', PROCESSED_DIR)
print('보고서 폴더:', REPORT_DIR)


Python 실행 파일: c:\dev\llm-data-analysis-study\llm-data-analysis-course\.venv\Scripts\python.exe
프로젝트 루트: C:\dev\llm-data-analysis-study\llm-data-analysis-course
전처리 데이터 폴더: C:\dev\llm-data-analysis-study\llm-data-analysis-course\data\processed
보고서 폴더: C:\dev\llm-data-analysis-study\llm-data-analysis-course\reports


## 2. 전처리 데이터 확인과 불러오기

이 장에서는 `customers_clean.csv`, `orders_clean.csv`, `order_items_clean.csv`를 사용합니다.


In [2]:
required_files = [
    PROCESSED_DIR / 'customers_clean.csv',
    PROCESSED_DIR / 'orders_clean.csv',
    PROCESSED_DIR / 'order_items_clean.csv',
]

missing_files = [path for path in required_files if not path.exists()]
if missing_files:
    raise FileNotFoundError(
        '전처리 파일이 없습니다: '
        + ', '.join(str(path) for path in missing_files)
        + '. 먼저 python scripts/preprocess_data.py를 실행하세요.'
    )

customers = pd.read_csv(PROCESSED_DIR / 'customers_clean.csv')
orders = pd.read_csv(PROCESSED_DIR / 'orders_clean.csv')
order_items = pd.read_csv(PROCESSED_DIR / 'order_items_clean.csv')

print('customers:', customers.shape, customers.columns.tolist())
print('orders:', orders.shape, orders.columns.tolist())
print('order_items:', order_items.shape, order_items.columns.tolist())


customers: (150, 6) ['customer_id', 'name', 'gender', 'age', 'city', 'signup_date']
orders: (300, 7) ['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status', 'order_month', 'order_dayofweek']
order_items: (764, 6) ['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price', 'line_total']


## 3. 주문 상태와 타깃 범위 확인

이번 실습은 완료 주문과 취소 주문만 비교합니다.

- `completed` → 0
- `cancelled` → 1
- `refunded`와 기타 상태 → 학습 대상에서 제외


In [3]:
status_summary = (
    orders['order_status']
    .value_counts(dropna=False)
    .rename_axis('order_status')
    .reset_index(name='order_count')
)
status_summary


,order_status,order_count
0,completed,184
1,cancelled,64
2,refunded,52


## 4. 분류 데이터와 병합 검증표 생성

공통 함수는 주문 특징 집계, 안전한 병합, 시간 순서 점검, 타깃 생성까지 수행합니다.


In [4]:
from src.classification import (
    build_classification_dataset,
    target_distribution,
)

(
    model_data,
    numeric_features,
    categorical_features,
    merge_checks,
    data_quality_checks,
) = build_classification_dataset(
    customers=customers,
    orders=orders,
    order_items=order_items,
)

target_dist = target_distribution(model_data)

print('모델링 데이터:', model_data.shape)
print('숫자형 feature:', numeric_features)
print('범주형 feature:', categorical_features)
display(target_dist)
display(merge_checks)
display(data_quality_checks)


ValueError: signup_date가 order_date보다 늦은 주문이 있습니다: 45건

In [5]:
chk = orders.merge(customers[["customer_id", "signup_date"]], on="customer_id", how="left")
chk["order_date"] = pd.to_datetime(chk["order_date"], errors="coerce")
chk["signup_date"] = pd.to_datetime(chk["signup_date"], errors="coerce")
chk["gap_days"] = (chk["order_date"] - chk["signup_date"]).dt.days

bad = chk[chk["gap_days"] < 0]
print("불일치 건수:", len(bad), "/ 전체", len(chk))
print("날짜 파싱 실패:", chk["order_date"].isna().sum(), chk["signup_date"].isna().sum())
print(chk[["order_date", "signup_date"]].agg(["min", "max"]))
print(bad["order_status"].value_counts())
print(bad["gap_days"].describe())
print(bad[["order_id", "customer_id", "order_date", "signup_date", "gap_days"]].head(10))

불일치 건수: 56 / 전체 300
날짜 파싱 실패: 0 0
    order_date signup_date
min 2025-09-16  2023-09-21
max 2026-09-15  2026-09-04
order_status
completed    35
refunded     11
cancelled    10
Name: count, dtype: int64
count     56.000000
mean    -139.696429
std       97.489565
min     -334.000000
25%     -214.250000
50%     -122.000000
75%      -49.750000
max       -3.000000
Name: gap_days, dtype: float64
    order_id  customer_id order_date signup_date  gap_days
6          7          120 2026-02-05  2026-04-24       -78
10        11           47 2026-01-25  2026-02-19       -25
12        13          117 2026-08-10  2026-09-01       -22
18        19            6 2026-02-02  2026-09-04      -214
24        25          148 2025-11-13  2026-08-23      -283
31        32          111 2026-05-12  2026-07-27       -76
48        49          104 2025-10-18  2026-09-03      -320
52        53          117 2025-12-18  2026-09-01      -257
54        55           50 2026-03-14  2026-06-27      -105
58        59     

## 추가 셀: 데이터 품질 문제 처리 (가입일 > 주문일)

**추가한 이유**
- 섹션 4의 `build_classification_dataset`이 `ValueError: signup_date가 order_date보다 늦은 주문이 있습니다: 45건`으로 중단되었다.
- 진단 결과 `orders`와 `customers`를 병합하면 전체 300건 중 56건(completed 35, cancelled 10, refunded 11)에서 가입일이 주문일보다 3~334일 늦었다. 날짜 변환 실패는 0건이다.
- 원본 저장소를 새로 clone해서 확인해도 같은 56건이 나와서, 로컬 환경 문제가 아니라 배포된 데이터 자체의 문제로 판단했다.
- 가이드와 `prepare_ch10_data.py`에는 이 경우의 처리 지침이 없다. `days_since_signup`은 가이드 STEP 1에서 사용 후보 특징으로 정해져 있고, 코드는 음수 일수가 있으면 중단한다.

**처리 방식 (임시 판단)**
- `signup_date`가 `order_date`보다 늦은 주문 56건을 별도 사본 폴더(`data/processed_ch10_filtered`)에서 제외한다.
- 해당 주문의 `order_items`도 함께 제외한다.
- 원본 `data/processed`와 `src/classification.py`는 수정하지 않는다.
- 이후 셀은 이 사본 폴더(`PROCESSED_DIR`)를 사용한다.

**한계**
- 제외한 행이 우연히 선택된 것이 아닐 수 있어 결과가 일부 편향될 수 있다. 어느 쪽이 맞는지(가입일 오류인지 주문일 오류인지)는 확인하지 못했다.

In [5]:
import pandas as pd

dst_dir = PROJECT_ROOT / "data" / "processed_ch10_filtered"
dst_dir.mkdir(exist_ok=True)

c = pd.read_csv(PROCESSED_DIR / "customers_clean.csv")
p = pd.read_csv(PROCESSED_DIR / "products_clean.csv")
o = pd.read_csv(PROCESSED_DIR / "orders_clean.csv")
i = pd.read_csv(PROCESSED_DIR / "order_items_clean.csv")

m = o.merge(c[["customer_id", "signup_date"]], on="customer_id", how="left")
bad_ids = m.loc[
    pd.to_datetime(m["order_date"]) < pd.to_datetime(m["signup_date"]), "order_id"
]

o2 = o[~o["order_id"].isin(bad_ids)]
i2 = i[~i["order_id"].isin(bad_ids)]

print("제외 주문:", len(bad_ids), "| orders:", len(o), "->", len(o2),
      "| order_items:", len(i), "->", len(i2))
print(o2["order_status"].value_counts())

c.to_csv(dst_dir / "customers_clean.csv", index=False)
p.to_csv(dst_dir / "products_clean.csv", index=False)
o2.to_csv(dst_dir / "orders_clean.csv", index=False)
i2.to_csv(dst_dir / "order_items_clean.csv", index=False)

PROCESSED_DIR = dst_dir
print("PROCESSED_DIR ->", PROCESSED_DIR)

제외 주문: 56 | orders: 300 -> 244 | order_items: 764 -> 624
order_status
completed    149
cancelled     54
refunded      41
Name: count, dtype: int64
PROCESSED_DIR -> C:\dev\llm-data-analysis-study\llm-data-analysis-course\data\processed_ch10_filtered


In [ ]:
# 섹션 2 재실행

required_files = [
    PROCESSED_DIR / 'customers_clean.csv',
    PROCESSED_DIR / 'orders_clean.csv',
    PROCESSED_DIR / 'order_items_clean.csv',
]

missing_files = [path for path in required_files if not path.exists()]
if missing_files:
    raise FileNotFoundError(
        '전처리 파일이 없습니다: '
        + ', '.join(str(path) for path in missing_files)
        + '. 먼저 python scripts/preprocess_data.py를 실행하세요.'
    )

customers = pd.read_csv(PROCESSED_DIR / 'customers_clean.csv')
orders = pd.read_csv(PROCESSED_DIR / 'orders_clean.csv')
order_items = pd.read_csv(PROCESSED_DIR / 'order_items_clean.csv')

print('customers:', customers.shape, customers.columns.tolist())
print('orders:', orders.shape, orders.columns.tolist())
print('order_items:', order_items.shape, order_items.columns.tolist())


customers: (150, 6) ['customer_id', 'name', 'gender', 'age', 'city', 'signup_date']
orders: (244, 7) ['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status', 'order_month', 'order_dayofweek']
order_items: (624, 6) ['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price', 'line_total']


In [7]:
# 섹션 3 재실행

status_summary = (
    orders['order_status']
    .value_counts(dropna=False)
    .rename_axis('order_status')
    .reset_index(name='order_count')
)
status_summary


,order_status,order_count
0,completed,149
1,cancelled,54
2,refunded,41


In [9]:
# 섹션 4 재실행

from src.classification import (
    build_classification_dataset,
    target_distribution,
)

(
    model_data,
    numeric_features,
    categorical_features,
    merge_checks,
    data_quality_checks,
) = build_classification_dataset(
    customers=customers,
    orders=orders,
    order_items=order_items,
)

target_dist = target_distribution(model_data)

print('모델링 데이터:', model_data.shape)
print('숫자형 feature:', numeric_features)
print('범주형 feature:', categorical_features)
display(target_dist)
display(merge_checks)
display(data_quality_checks)

모델링 데이터: (203, 16)
숫자형 feature: ['age', 'item_count', 'total_quantity', 'order_amount', 'order_month', 'order_dayofweek', 'days_since_signup']
범주형 feature: ['gender', 'city', 'payment_method']


,is_cancelled,class_label,count,ratio
0,0,completed,149,0.734
1,1,cancelled,54,0.266


,merge,before_rows,after_rows,row_count_preserved,unmatched_count,status
0,order_id → order_item_features,203,203,True,0,PASS
1,customer_id → customers,203,203,True,0,PASS


,check_item,count
0,binary_target_rows,203
1,completed_rows,149
2,cancelled_rows,54
3,excluded_status_rows,41
4,line_total_mismatch,0
5,negative_days_since_signup,0
6,missing_order_item_features,0
7,missing_customer_match,0
8,status_scope:completed,149
9,status_scope:cancelled,54


## 5. 데이터 누수 확인

`order_status`와 `is_cancelled`는 입력 feature에 포함되면 안 됩니다.


In [10]:
features = numeric_features + categorical_features
leakage_columns = {'order_status', 'is_cancelled'}
leakage_found = sorted(leakage_columns.intersection(features))

if leakage_found:
    raise ValueError(f'데이터 누수 위험 컬럼이 포함되었습니다: {leakage_found}')

print('사용 feature:', features)
print('데이터 누수 컬럼 없음')


사용 feature: ['age', 'item_count', 'total_quantity', 'order_amount', 'order_month', 'order_dayofweek', 'days_since_signup', 'gender', 'city', 'payment_method']
데이터 누수 컬럼 없음


## 6. train, validation, test 분리

모델과 임계값은 validation에서 선택하고, test는 최종 평가에 한 번만 사용합니다.


In [11]:
from src.classification import (
    split_train_validation_test,
    build_split_summary,
)

(
    X_train,
    X_valid,
    X_test,
    y_train,
    y_valid,
    y_test,
    features,
) = split_train_validation_test(
    model_data=model_data,
    numeric_features=numeric_features,
    categorical_features=categorical_features,
    random_state=42,
)

split_summary = build_split_summary(y_train, y_valid, y_test)

print('X_train:', X_train.shape)
print('X_valid:', X_valid.shape)
print('X_test:', X_test.shape)
display(split_summary)


X_train: (121, 10)
X_valid: (41, 10)
X_test: (41, 10)


,split,is_cancelled,class_label,count,ratio
0,train,0,completed,89,0.7355
1,train,1,cancelled,32,0.2645
2,validation,0,completed,30,0.7317
3,validation,1,cancelled,11,0.2683
4,test,0,completed,30,0.7317
5,test,1,cancelled,11,0.2683


## 7. 기준 모델과 학습 모델 비교

가장 많은 클래스만 예측하는 Dummy 모델을 기준으로 Logistic Regression과 Random Forest의 validation 성능을 비교합니다.


In [12]:
from src.classification import train_and_compare_on_validation

(
    models,
    validation_comparison,
    validation_predictions,
    validation_probabilities,
) = train_and_compare_on_validation(
    X_train=X_train,
    X_valid=X_valid,
    y_train=y_train,
    y_valid=y_valid,
    numeric_features=numeric_features,
    categorical_features=categorical_features,
    random_state=42,
)

validation_comparison


,model,evaluation_split,accuracy,precision,recall,f1
0,Logistic Regression,validation,0.634146,0.400000,0.727273,0.516129
1,Random Forest,validation,0.658537,0.285714,0.181818,0.222222
2,Dummy Most Frequent,validation,0.731707,0.000000,0.000000,0.000000


## 8. validation에서 모델과 임계값 선택

Dummy 모델은 기준선이므로 실제 선택 후보에서는 제외합니다. validation의 f1-score를 우선하고 recall과 precision을 함께 확인합니다.


In [13]:
from src.classification import threshold_metrics, choose_threshold

non_dummy_comparison = validation_comparison[
    validation_comparison['model'] != 'Dummy Most Frequent'
]

selected_model_name = str(non_dummy_comparison.iloc[0]['model'])
selected_model = models[selected_model_name]
validation_proba = validation_probabilities[selected_model_name]

threshold_df = threshold_metrics(y_valid, validation_proba)
selected_threshold = choose_threshold(threshold_df)

print('선택 모델:', selected_model_name)
print('선택 임계값:', selected_threshold)
display(threshold_df.sort_values(['f1', 'recall'], ascending=False).head(10))


선택 모델: Logistic Regression
선택 임계값: 0.6


,threshold,accuracy,precision,recall,f1
8,0.60,0.780488,0.583333,0.636364,0.608696
5,0.45,0.658537,0.428571,0.818182,0.562500
7,0.55,0.682927,0.444444,0.727273,0.551724
6,0.50,0.634146,0.400000,0.727273,0.516129
4,0.40,0.560976,0.360000,0.818182,0.500000
9,0.65,0.756098,0.555556,0.454545,0.500000
10,0.70,0.804878,0.800000,0.363636,0.500000
11,0.75,0.804878,0.800000,0.363636,0.500000
1,0.25,0.463415,0.322581,0.909091,0.476190
2,0.30,0.463415,0.322581,0.909091,0.476190


## 9. test 데이터 최종 평가

이제 선택이 끝났으므로 test 데이터에서 최종 성능을 한 번 확인합니다.


In [14]:
from src.classification import (
    final_test_evaluation,
    confusion_matrix_dataframe,
    classification_report_dataframe,
)

y_pred_test, y_proba_test, test_metrics = final_test_evaluation(
    selected_model,
    X_test,
    y_test,
    threshold=selected_threshold,
)

confusion_df = confusion_matrix_dataframe(y_test, y_pred_test)
classification_report_df = classification_report_dataframe(y_test, y_pred_test)

display(test_metrics)
display(confusion_df)
display(classification_report_df)


,evaluation_split,threshold,selection_status,accuracy,precision,recall,f1
0,test,0.6,frozen_before_test,0.682927,0.4,0.363636,0.380952


,pred_completed,pred_cancelled
actual_completed,24,6
actual_cancelled,7,4


,label,precision,recall,f1-score,support
0,completed,0.774194,0.800000,0.786885,30.000000
1,cancelled,0.400000,0.363636,0.380952,11.000000
2,accuracy,0.682927,0.682927,0.682927,0.682927
3,macro avg,0.587097,0.581818,0.583919,41.000000
4,weighted avg,0.673800,0.682927,0.677976,41.000000


혼동행렬은 다음처럼 읽습니다.

| 구분 | 의미 |
|---|---|
| True Negative | 실제 완료 주문을 완료로 예측 |
| False Positive | 실제 완료 주문을 취소로 잘못 예측 |
| False Negative | 실제 취소 주문을 완료로 잘못 예측 |
| True Positive | 실제 취소 주문을 취소로 예측 |


## 10. 개인정보를 제외한 예측 결과 생성

고객명, 이메일, 주소 같은 불필요한 식별정보는 결과 파일에 저장하지 않습니다.


In [15]:
from src.classification import create_prediction_result

prediction_result = create_prediction_result(
    source_index=X_test.index,
    y_test=y_test,
    y_pred=y_pred_test,
    y_proba=y_proba_test,
    model_name=selected_model_name,
    threshold=selected_threshold,
)

prediction_result.head()


,record_id,source_index,actual_is_cancelled,predicted_is_cancelled,cancel_probability,model,threshold
0,test_0001,79,0,0,0.569824,Logistic Regression,0.6
1,test_0002,20,0,0,0.576583,Logistic Regression,0.6
2,test_0003,104,0,0,0.328096,Logistic Regression,0.6
3,test_0004,119,1,1,0.880826,Logistic Regression,0.6
4,test_0005,29,0,0,0.545805,Logistic Regression,0.6


## 11. LLM 코드 검토 체크리스트


In [16]:
from src.classification import build_classification_checklist

classification_checklist = build_classification_checklist()
classification_checklist


,check_item,status
0,completed와 cancelled만 사용해 이진 타깃을 만들었는가?,□
1,refunded 등 다른 상태를 0 클래스에 섞지 않았는가?,□
2,예측 시점과 feature 가용성 가정을 설명할 수 있는가?,□
3,line_total = quantity × unit_price를 검증했는가?,□
4,병합에 validate를 사용하고 미매칭 0건을 확인했는가?,□
5,"order_status, target, ID, 사후 정보를 feature에서 제외했는가?",□
6,"train, validation, test를 분리했는가?",□
7,모델과 threshold는 validation에서 선택했는가?,□
8,모델과 threshold를 고정한 뒤 test를 한 번만 사용했는가?,□
9,Dummy 기준 모델과 비교했는가?,□


## 12. 전체 파이프라인 실행과 산출물 저장

위 과정을 한 번에 다시 실행해 CSV와 Markdown 보고서를 생성합니다.


In [17]:
from src.classification import run_classification_analysis

classification_result = run_classification_analysis(
    processed_dir=PROCESSED_DIR,
    report_dir=REPORT_DIR,
    random_state=42,
)

print('선택 모델:', classification_result['selected_model_name'])
print('선택 임계값:', classification_result['selected_threshold'])
display(classification_result['test_metrics'])

for name, path in classification_result['output_paths'].items():
    print(name, path, 'OK' if path.exists() else 'MISSING')


선택 모델: Logistic Regression
선택 임계값: 0.6


,evaluation_split,threshold,selection_status,accuracy,precision,recall,f1
0,test,0.6,frozen_before_test,0.682927,0.4,0.363636,0.380952


model_data_internal C:\dev\llm-data-analysis-study\llm-data-analysis-course\reports\ch10_classification_model_data_internal.csv OK
target_distribution C:\dev\llm-data-analysis-study\llm-data-analysis-course\reports\ch10_target_distribution.csv OK
feature_audit C:\dev\llm-data-analysis-study\llm-data-analysis-course\reports\ch10_feature_audit.csv OK
merge_checks C:\dev\llm-data-analysis-study\llm-data-analysis-course\reports\ch10_merge_checks.csv OK
data_quality_checks C:\dev\llm-data-analysis-study\llm-data-analysis-course\reports\ch10_data_quality_checks.csv OK
split_summary C:\dev\llm-data-analysis-study\llm-data-analysis-course\reports\ch10_split_summary.csv OK
validation_comparison C:\dev\llm-data-analysis-study\llm-data-analysis-course\reports\ch10_validation_model_comparison.csv OK
threshold_metrics C:\dev\llm-data-analysis-study\llm-data-analysis-course\reports\ch10_validation_threshold_metrics.csv OK
test_metrics C:\dev\llm-data-analysis-study\llm-data-analysis-course\reports\c

## 13. LLM 검토 프롬프트 예시

```text
온라인 쇼핑몰 주문 취소 분류 코드를 검토해 주세요.

타깃 기준:
- completed=0
- cancelled=1
- refunded와 기타 상태는 제외

검토 항목:
1. order_status와 is_cancelled가 feature에서 제외되었는가?
2. 병합에 validate와 indicator가 사용되었는가?
3. train, validation, test가 분리되었는가?
4. 모델과 임계값은 validation에서 선택했는가?
5. test는 최종 평가에 한 번만 사용했는가?
6. Dummy 기준 모델보다 나은가?
7. accuracy, precision, recall, f1을 함께 확인했는가?
8. 예측 결과에 개인정보가 포함되지 않았는가?
9. 모델 결과를 취소 원인으로 단정하지 않았는가?

필수 수정과 권장 개선을 구분해 설명해 주세요.
```


## 14. 정리

이번 장에서는 완료 주문과 취소 주문을 명확하게 구분하고, 병합 검증, 데이터 누수 방지, train/validation/test 분리, 기준 모델 비교, validation 기반 모델·임계값 선택, 최종 test 평가를 수행했습니다.

다음 장에서는 LLM을 활용해 분석 질문과 프롬프트를 더 구체적으로 만드는 방법으로 이어집니다.


## 제출 정보
- 이름: 권혁수
- GitHub ID: kwonhyeoksoo
- 작성일: 2026-11-18
- 최종 Notebook URL: https://github.com/kwonhyeoksoo/llm-data-analysis-study/blob/main/chapter10/chapter10.ipynb

## 1. 분류 문제 정의
- 타깃: `is_cancelled`
- 0의 의미: `completed` (정상 완료된 주문)
- 1의 의미: `cancelled` (취소된 주문)
- 제외한 상태: `refunded` 41건 (취소로도 완료로도 분류하지 않고 모델링에서 제외)
- 예측 단위: 주문 1건
- 예측 시점: 주문 생성 직후 (교육용 가정)
- 예측 시점에 사용할 수 있는 정보: `age`, `gender`, `city`, `payment_method`, `item_count`, `total_quantity`, `order_amount`, 주문 월(`order_month`), 주문 요일(`order_dayofweek`), `days_since_signup`
- 예측 이후에만 알 수 있어 제외한 정보: `order_status`, `is_cancelled`, `cancel_reason`, `cancelled_at`, 취소 이후 생성되는 정보, 그리고 식별자(`order_id`, `customer_id`, `product_id`)

### 나의 해석과 판단
`completed`와 `cancelled`만 타깃으로 삼은 이유는 두 상태만 '주문이 정상 완료되었는가, 취소되었는가'라는 이진 질문에 직접 대응하기 때문이다. `refunded`는 취소와 의미가 다르고 어느 쪽인지 근거가 없어서 0이나 1로 자동 변환하지 않고 제외했다. 예측 시점은 주문 생성 직후로 정했다. 이 시점에 이미 알 수 있는 정보만 특징으로 써야 실제 운영에서 같은 입력을 만들 수 있다. 0은 단순한 비취소가 아니라 `completed`라는 점을 분명히 했다.

---

## 2. Feature Contract와 Leakage Audit

### 사용한 숫자형 Feature
`age`, `item_count`, `total_quantity`, `order_amount`, `order_month`, `order_dayofweek`, `days_since_signup` (7개)

### 사용한 범주형 Feature
`gender`, `city`, `payment_method` (3개)

### 금지 Feature

```text
order_status
is_cancelled
order_id
customer_id
product_id
취소 이후 정보
```

Feature Audit 결과 금지 사유는 다음과 같았다. `cancel_reason`, `cancelled_at`은 취소 이후 생성되는 사후 정보, `customer_id`, `order_id`, `product_id`는 식별자, `is_cancelled`는 예측 대상 자체, `order_status`는 예측 결과와 직접 연결되는 주문 상태이다. 자동 점검에서 금지 컬럼과 사용 특징의 겹침은 0건이었다.

### 나의 해석과 판단
파일에 컬럼이 있다는 것은 데이터에 존재한다는 뜻일 뿐, 예측하는 순간에 알 수 있다는 뜻이 아니다. 예를 들어 `order_status`나 `cancelled_at`은 취소가 일어난 뒤에야 값이 생기므로 이를 입력에 넣으면 정답을 미리 보고 맞히는 것과 같아 성능이 실제보다 좋게 보인다. 따라서 이 값이 예측하는 순간에 실제로 알 수 있는지를 기준으로 특징을 골랐다.

---

## 3. 주문 단위 특징과 병합 검증

### 계산 관계 확인

```text
line_total = quantity × unit_price
```

- 계산 불일치 건수: 0건 (`line_total_mismatch` = 0)
- 주문 특징 `order_id` 중복 건수: 0건 (주문 ↔ 주문 특징 병합을 `one_to_one`으로 검증해 통과)

### 병합 검증

| 병합 | 관계 | 병합 전 행 수 | 병합 후 행 수 | 미매칭 | 판정 |
| --- | --- | ---: | ---: | ---: | --- |
| 주문 ↔ 주문 특징 | one_to_one | 203 | 203 | 0 | PASS |
| 주문 ↔ 고객 | many_to_one | 203 | 203 | 0 | PASS |

### 결과 관찰
두 병합 모두 행 수가 유지되었고 미매칭이 0건이었다. `missing_order_item_features`와 `missing_customer_match`도 0이었다.

데이터 품질 문제가 하나 있었다. 처음 실행했을 때 `build_classification_dataset`이 "signup_date가 order_date보다 늦은 주문이 있습니다: 45건"이라는 오류로 중단되었다. 확인해 보니 전체 주문 300건 중 56건(completed 35, cancelled 10, refunded 11)에서 가입일이 주문일보다 3~334일 늦었다(날짜 변환 실패는 0건). 저장소를 새로 받아 확인해도 같은 56건이 나와 배포된 데이터 자체의 문제로 판단했다. 가이드에 처리 지침이 없었고 이와 관련하여 교수님의 피드백을 과제 기한 내에 받지 못했기 때문에 임시로 이 56건과 해당 주문의 `order_items`를 별도 사본 폴더에서 제외했다. 그 결과 주문은 300건에서 244건, `order_items`는 764건에서 624건이 되었고, 모델링 대상은 completed 149 + cancelled 54 = 203건이 되었다. 원본 데이터와 `src/classification.py`는 수정하지 않았다.

### 나의 해석과 판단
병합에서 미매칭이 생겼을 때 원인을 확인하지 않고 0으로 채우면 실제로는 정보가 없는 것을 '값이 0인 것'으로 바꿔 버리기 때문에 분석 결과가 왜곡되므로 병합마다 `validate`로 관계를 검증하고 미매칭 건수를 확인했다. 가입일 문제도 같은 이유로 값을 임의로 고치지 않고 제외 기준과 건수를 기록하는 방식으로 처리했다. 제외한 행이 우연히 선택된 것이 아닐 수 있어 결과가 일부 편향될 수 있고 가입일과 주문일 중 어느 쪽이 잘못된 것인지는 확인하지 못했다.

---

## 4. 클래스 비율

| 클래스 | 의미 | 개수 | 비율 |
| ---: | --- | ---: | ---: |
| 0 | completed | 149 | 73.4% |
| 1 | cancelled | 54 | 26.6% |

![클래스 비율](images/step02_class_ratio.png)

### 결과 관찰
모델링 대상 203건 중 취소가 54건(26.6%)으로 소수 클래스이다.

### 나의 해석과 판단
취소가 약 4분의 1이라서 모든 주문을 completed로 예측해도 Accuracy가 약 73%가 나오므로 Accuracy만 보면 취소를 하나도 못 잡는 모델도 좋아 보일 수 있다. 취소를 얼마나 잡았는지는 Precision, Recall, F1로 따로 봐야 한다.

---

## 5. Dummy Baseline과 Validation 모델 비교

| 모델 | Accuracy | Precision | Recall | F1 |
| --- | ---: | ---: | ---: | ---: |
| Dummy Most Frequent | 0.732 | 0.000 | 0.000 | 0.000 |
| Logistic Regression | 0.634 | 0.400 | 0.727 | 0.516 |
| Random Forest | 0.659 | 0.286 | 0.182 | 0.222 |

(Validation 41건, 기본 Threshold 0.5 기준)

![Baseline 비교](images/step03_baseline.png)

- Validation에서 선택한 모델: Logistic Regression (Validation F1 기준 1위)

### 결과 관찰
Dummy는 Accuracy가 0.732로 가장 높지만 취소를 단 한 개도 잡지 못해 Recall과 F1이 0이다. Logistic Regression은 Accuracy가 더 낮지만 취소 11건 중 대부분을 잡아 Recall 0.727, F1 0.516이다. Random Forest는 Recall이 0.182로 낮다.

### 나의 해석과 판단
Logistic Regression이 Dummy보다 나아진 점은 취소를 실제로 잡아낸다는 것이다(Recall 0 → 0.727, F1 0 → 0.516). 반대로 Accuracy는 Dummy보다 낮아졌다. 따라서 Accuracy가 더 높은 Dummy가 더 좋다고 볼 수는 없고 이 문제에서는 Accuracy가 모델 비교에 부적절한 지표이다.

### 한계와 추가 확인 사항
Validation은 41건이고 취소는 11건뿐이라 몇 건만 달라져도 지표가 크게 바뀐다. 모델 간 차이가 우연일 가능성을 배제하지 못하므로 반복 검증이나 더 큰 표본이 필요하다.

---

## 6. Train / Validation / Test 역할

| Split | 역할 | 클래스 0 건수 | 클래스 1 건수 |
| --- | --- | ---: | ---: |
| Train | 모델 학습 | 89 | 32 |
| Validation | 모델·Threshold 선택 | 30 | 11 |
| Test | 선택 이후 최종 평가 | 30 | 11 |

(합계 203건, `random_state=42`, 층화 랜덤 분할. 세 분할의 취소 비율은 26.5~26.8%로 비슷했다.)

### 나의 해석과 판단
Test를 모델이나 Threshold 선택에 쓰면 같은 데이터로 고르고 같은 데이터로 평가하는 셈이라 최종 성능이 실제보다 좋게 나온다. Test는 선택을 모두 끝낸 뒤 한 번만 보고, 보고 난 뒤에는 선택을 바꾸지 않아야 새 데이터에서의 성능을 대략 가늠할 수 있다.

### 교육용 random split의 한계
이번 분할은 시간 순서를 고려하지 않은 랜덤 분할이다. 실제 운영에서는 과거 주문으로 학습해 미래 주문을 예측하므로 사용 전에 시간 순서로 나눈 평가(out-of-time 검증)가 필요하다. 이와 함께 더 긴 기간의 데이터, 월별 성능 변화 확인, 표본 수 확대가 필요하다.

---

## 7. Validation 성능과 우선 지표

선택한 Logistic Regression, Threshold 0.6 기준 Validation 성능이다.

- Accuracy: 0.780
- Precision: 0.583
- Recall: 0.636
- F1: 0.609

(참고: 기본 Threshold 0.5에서는 Accuracy 0.634, Precision 0.400, Recall 0.727, F1 0.516)

### 내가 가장 중요하게 본 지표
F1 (Precision과 Recall의 균형)

### 그 이유
취소를 놓치는 오류(FN)와 정상 주문을 취소 위험으로 잘못 보는 오류(FP)를 모두 고려하기 위해서이다. 이번 실습의 Threshold 선택도 Validation F1이 가장 높은 값을 기준으로 했다.

> 실제 FP/FN 비용 정보가 없다. F1을 우선한 것은 두 오류를 비슷한 비중으로 본다는 가정이며 실제 비용을 알게 되면 우선 지표를 바꿀 수 있다.

---

## 8. Threshold 비교

Validation에서 0.20~0.80(0.05 간격, 13개) 후보를 비교했다. 대표 값은 다음과 같다.

| Threshold | Precision | Recall | F1 | 예상 특징 |
| ---: | ---: | ---: | ---: | --- |
| 0.45 | 0.429 | 0.818 | 0.563 | 취소를 더 많이 잡지만 오경보가 많음 |
| **0.60** | 0.583 | 0.636 | **0.609** | Precision과 Recall이 비교적 균형 (F1 최대) |
| 0.70 | 0.800 | 0.364 | 0.500 | 예측한 취소는 정확하지만 놓치는 취소가 많음 |

(기본값 0.50: Precision 0.400, Recall 0.727, F1 0.516)

![Threshold 비교](images/step05_threshold.png)

### 내가 선택한 Threshold
0.6

### 선택 이유
후보 13개 중 Validation F1이 가장 높았다(0.609). Threshold를 올리면 Precision은 오르고 Recall은 내려가며, 내리면 반대가 된다. 다만 Validation 취소가 11건뿐이라 0.60과 0.45의 F1 차이(0.609 대 0.563)는 주문 한두 건 차이이다. 따라서 이 선택이 안정적이라고 단정하지 않는다.

### 나의 해석과 판단
Threshold도 모델과 마찬가지로 데이터를 보고 정하는 값이다. Final Test에서 정하면 Test에 맞춰진 값이 되어 최종 평가가 부풀려진다. 그래서 Validation에서 정하고 Test 전에 고정했다.

---

## 9. FP/FN 해석

- FP 의미: 실제로는 완료된 주문(`completed`)을 취소 위험으로 잘못 예측한 경우 (Test 6건)
- FN 의미: 실제로 취소된 주문(`cancelled`)을 완료될 것으로 잘못 예측해 놓친 경우 (Test 7건)
- 현재 목적에서 더 부담스러울 수 있는 오류: 취소를 놓치는 FN일 수 있다고 가정하지만 업무 비용 정보가 없어 확정하지 않는다.
- 그 판단에 추가로 필요한 업무 정보: 취소 주문 한 건의 손실, 취소 위험 주문에 대한 대응 비용(확인 연락 등), 오경보로 정상 고객이 겪는 불편, 대응 가능한 인력과 시간

| | 예측 completed | 예측 cancelled |
| --- | ---: | ---: |
| 실제 completed | 24 (TN) | 6 (FP) |
| 실제 cancelled | 7 (FN) | 4 (TP) |

![Confusion Matrix](images/step06_confusion_matrix.png)

### 업무·분석적 의미
Test에서 실제 취소 11건 중 4건만 잡고 7건을 놓쳤으며 취소로 예측한 10건 중 맞힌 것은 4건이다. 현재 Threshold에서는 취소를 놓치는 비중이 크고 취소 위험으로 표시된 주문의 절반 이상이 실제로는 취소되지 않았다.

### 한계와 추가 확인 사항
Test 취소가 11건뿐이라 혼동행렬의 숫자 몇 개가 달라지면 결과 해석이 달라진다. FP와 FN 중 어느 쪽이 더 큰 문제인지는 비용 정보 없이 판단할 수 없다.

---

## 10. Final Test

- 고정한 모델: Logistic Regression
- 고정한 Threshold: 0.6
- Accuracy: 0.683
- Precision: 0.400
- Recall: 0.364
- F1: 0.381
- Validation 대비 변화: Accuracy 0.780 → 0.683, Precision 0.583 → 0.400, Recall 0.636 → 0.364, F1 0.609 → 0.381 (모두 하락)

### 결과 관찰
Test 성능은 Validation보다 전반적으로 낮았다. 참고로 Test는 completed가 30건이라 전부 completed로 예측하면 Accuracy가 30/41 ≈ 0.732(계산)이어서 이번 모델의 Accuracy 0.683보다 높지만,  그 방식은 취소를 하나도 잡지 못한다(Recall 0).

### 나의 해석과 판단
Validation에서 좋아 보였던 성능이 Test에서 유지되지 않아 이 모델이 새 데이터에서도 같은 성능을 낸다고 믿기 어렵다. 가능한 설명으로는 표본이 작아 지표가 불안정한 점, Validation에서 Threshold를 고르는 과정에서 Validation에 맞춰졌을 가능성이 있다. 이 실습만으로 원인을 확정할 수는 없기 때문에 이 결과를 본 뒤에도 모델이나 Threshold를 바꾸지 않고 그대로 기록했다.

### 선택 보호 확인
- [x] Test 결과를 보기 전에 모델을 고정했습니다.
- [x] Test 결과를 보기 전에 Threshold를 고정했습니다.
- [x] Test 결과를 본 뒤 같은 Test를 이용해 선택을 바꾸지 않았습니다.

---

## 11. Internal / Public 결과 구분

### 공개 Prediction 컬럼

```text
record_id
actual_is_cancelled
predicted_is_cancelled
cancel_probability
model
threshold
```

### 공개 결과에 없어야 할 컬럼

```text
source_index
order_id
customer_id
product_id
```

### 결과 관찰
노트북 섹션 10의 `prediction_result`는 `source_index`를 포함한 내부용 표였다. `public_prediction_result()`로 이를 제거한 공개용 `ch10_classification_predictions.csv`는 위의 6개 컬럼만 가지고 있었다. 내부용은 `ch10_classification_predictions_internal.csv`와 `ch10_classification_model_data_internal.csv`로 따로 저장되었다. 자동 검증의 `public_prediction_identifier_columns`는 `none`, PASS였다.

### 나의 해석과 판단
모델 오류를 분석하려면 어떤 원본 행에서 틀렸는지 추적할 수 있어야 하므로 `source_index` 같은 값이 든 내부 자료가 필요하다. 하지만 이런 값이 공개 결과에 있으면 특정 주문이나 고객을 다시 찾아낼 수 있으므로 내부용과 공개용을 파일 단위로 분리했다.

---

## 12. Validation Evidence

다음 Evidence 중 확인한 항목에 체크합니다.

- [x] `ch10_target_distribution.csv`
- [x] `ch10_feature_audit.csv`
- [x] `ch10_merge_checks.csv`
- [x] `ch10_data_quality_checks.csv`
- [x] `ch10_split_summary.csv`
- [x] `ch10_validation_model_comparison.csv`
- [x] `ch10_validation_threshold_metrics.csv`
- [x] `ch10_test_metrics.csv`
- [x] `ch10_confusion_matrix.csv`
- [x] `ch10_classification_validation.csv`

### 자동 Validation 결과

| 검증 항목 | 값 | 상태 |
| --- | --- | --- |
| target_exactly_completed_cancelled | [0, 1] | PASS |
| forbidden_feature_overlap | 0 | PASS |
| merge_rows_and_matches | True | PASS |
| all_splits_have_both_classes | True | PASS |
| model_selected_on_validation | True | PASS |
| threshold_selected_on_validation | True | PASS |
| public_prediction_identifier_columns | none | PASS |

노트북 실행 결과와 스크립트(`python -m scripts.run_classification_analysis`) 재실행 결과(선택 모델, Threshold, Test 지표, 혼동행렬, 검증 항목)가 동일했다. 다만 원본에 가입일 문제 행이 있어 `prepare_ch10_data`는 다시 실행하지 않고, 제외한 사본 데이터를 `data/processed`에 임시로 넣고 분석 스크립트만 재실행했다.

### 자동 PASS가 의미하지 않는 것
자동 PASS는 현재 코드가 정한 규칙(타깃 범위, 금지 특징, 병합 일치, 선택 순서, 공개 컬럼)을 지켰다는 뜻이다. 모델이 실제 운영에 적합한지, 고객군이나 지역 간에 공정한지, 시간이 지나도 성능이 유지되는지, 그리고 제외한 56건이 결과를 편향시키지 않았는지는 자동 검증으로 알 수 없다.

---

## 13. 최종 사용 판단

- [ ] 참고용 사용 가능
- [ ] 추가 검증 후 사용 가능
- [x] 현재 사용 보류

### 판단 근거
1. Final Test에서 취소 11건 중 4건만 잡았고(Recall 0.364), Precision 0.400, F1 0.381로 낮다. Validation(F1 0.609)에서 크게 떨어졌다.
2. Accuracy(0.683)는 전부 completed로 예측하는 방식(약 0.732)보다 낮고, Validation과 Test의 취소가 각각 11건뿐이라 성능 추정이 불안정하다.
3. 시간 순서 검증 없이 랜덤 분할만 했고, 가입일 > 주문일 데이터 56건을 임시 기준으로 제외했으며, FP/FN 비용 정보가 없다.

### 현재 모델의 가장 큰 위험
취소 주문을 놓칠 수 있는 정도(Test에서 취소 11건 중 7건)와 표본이 작아 성능 추정이 불안정하다는 점이다. 이 상태로 사용하면 결과를 실제보다 믿을 만하다고 오해할 수 있다.

### 다음 개선 우선순위
1. 시간 순서로 나눈 검증(out-of-time)과 더 많은 주문 데이터로 성능이 유지되는지 확인한다.
2. FP/FN 업무 비용을 확인하고 그에 맞춰 우선 지표와 Threshold를 다시 정한다.
3. 가입일 > 주문일 데이터 56건의 원인을 확인하고, 제외 여부가 결과에 미치는 영향을 비교한다.

### 예측과 인과 구분
현재 모델에서 확인한 것은 "주문 생성 시점의 특징(고객 정보, 결제수단, 주문 금액/수량, 주문 시기, 가입 후 경과일)으로 주문이 취소될지를 어느 정도 예측할 수 있는지"라는 **예측 패턴**이다. 이 특징이 취소를 일으킨다는 **원인 주장**은 할 수 없다. 모델은 특징과 취소가 함께 나타나는 경향을 학습했을 뿐이고, 개입 실험이나 별도의 인과 분석 없이는 원인을 확인하지 못한다.

---

## 최종 체크

- [x] `completed=0`, `cancelled=1` 타깃 범위를 정의했습니다.
- [x] `refunded`와 기타 상태를 제외했습니다.
- [x] 예측 시점과 Feature Contract를 정의했습니다.
- [x] target, ID, 사후 정보를 feature에서 제외했습니다.
- [x] `line_total = quantity × unit_price`를 검증했습니다.
- [x] 주문 단위 특징과 병합 관계를 검증했습니다.
- [x] 클래스 비율을 확인했습니다.
- [x] Dummy baseline과 비교했습니다.
- [x] Train / Validation / Test 역할을 구분했습니다.
- [x] 모델을 Validation에서 선택했습니다.
- [x] Threshold를 Validation에서 선택했습니다.
- [x] Final Test 전에 선택을 고정했습니다.
- [x] Precision / Recall / F1을 해석했습니다.
- [x] FP/FN의 의미를 설명했습니다.
- [x] Public prediction에서 식별자를 제거했습니다.
- [x] Validation Evidence를 확인했습니다.
- [x] random split의 한계를 기록했습니다.
- [x] 예측 패턴을 원인처럼 단정하지 않았습니다.
- [x] 최종 Notebook URL을 제출합니다.